# **Preprocesamiento de datos**

In [41]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator

In [42]:
dataset = pd.read_csv("data/raw.csv")
raw_dataset = dataset.copy()

# Nuevas columnas

### Turbo

Buscar "T", "turbo" en la columna "Motor". También podríamos buscar en la descripción, versión o el título (ahora es solo en "Motor")

In [43]:
def is_turbo(row):
    motor = str(row.get('Motor', ''))
    version = str(row.get('Versión', ''))
    combined = f"{motor} {version}"
    # Look for 'turbo' or typical turbo patterns
    if re.search(r'turbo', combined, re.IGNORECASE):
        return 1
    if re.search(r'(?:\d[\.,]?\d?\s*[tT]|TSI|TFSI|TURBO)', combined, re.IGNORECASE):
        return 1
    return 0

dataset['Turbo'] = dataset.apply(is_turbo, axis=1)

### Caballos de fuerza (cv)


In [44]:
def extract_cv(text):
    if not isinstance(text, str):
        return np.nan
    match = re.search(r'(\d+)\s*[cC][vV]\b', text)
    if match:
        return int(match.group(1))
    return np.nan

dataset['cv'] = dataset['Versión'].apply(extract_cv)

## Tracción

In [45]:
def extract_traccion(text):
    if not isinstance(text, str):
        return np.nan
    if re.search(r'4[xX]4|4wd|awd', text, re.IGNORECASE):
        return '4x4'
    if re.search(r'4[xX]2|2wd|fwd', text, re.IGNORECASE):
        return '4x2'
    return np.nan

dataset['Tracción'] = dataset['Versión'].apply(extract_traccion)

# Transformación de columnas

## **Motor**

### Buscar el tamaño del motor en "Versión" si no está en motor

Ejemplo: Fila 86 del archivo @data/raw.csv 
- `Motor` es **TURBOALIMENTADO** y `Versión` es **2.0 T ALLSPACE COMFORTLINE AT L17**

In [46]:
def extract_engine_size(version):
    # Look for patterns like "2.0", "1.6", etc. at the start of the version string
    if not isinstance(version, str):
        return None
    match = re.match(r'(\d\.\d)', version)
    if match:
        return match.group(1)
    return None

def fix_motor(row):
    motor = row['Motor']
    version = row['Versión']
    try:
        float(motor)
        is_number = True
    except (ValueError, TypeError):
        is_number = False
    if not is_number:
        engine_size = extract_engine_size(version)
        if engine_size:
            return engine_size
    return motor

dataset['Motor'] = dataset.apply(fix_motor, axis=1)

print(dataset.iloc[84])

idx                                                                       84
Marca                                                             Volkswagen
Modelo                                                                Tiguan
Año                                                                   2018.0
Versión                                    2.0 T ALLSPACE COMFORTLINE AT L17
Color                                                                 BLANCA
Tipo de combustible                                                    Nafta
Puertas                                                                  5.0
Transmisión                                                       Automática
Motor                                                                    2.0
Tipo de carrocería                                                       SUV
Con cámara de retroceso                                                  NaN
Kilómetros                                                           89000.0

## **Versión**

### Borrar el cilindraje.

Ejemplo: borrar 2.4 de "2.4 Sport At"

In [47]:
def remove_cilinder_size(version):
    if not isinstance(version, str):
        return version
    # Remove patterns like "1.0", "1,0", "2.0", "2,0", "1.6", "1,6", "2.4", "2,4", "1.5", "1,5", etc.
    # Allow optional "L" or "l" after the number, and optional space
    # Only match at the start or after a space, and only if followed by space or end or non-word char
    pattern = r'(^|\s)(\d[\.,]\d)\s*[Ll]?(?=\s|$|\W)'
    version = re.sub(pattern, ' ', version)
    # Remove turbo-related patterns: T, t, TSI, TFSI, TURBO (case-insensitive)
    version = re.sub(r'\b\d[\.,]?\d?\s*[tT]\b', ' ', version)
    version = re.sub(r'\b(TSI|TFSI|TURBO)\b', ' ', version, flags=re.IGNORECASE)
    # Remove extra spaces
    version = re.sub(' +', ' ', version).strip()
    return version

dataset['Versión'] = dataset['Versión'].apply(remove_cilinder_size)

### Borrar "at" y "mt"
(Manual/Automatic transmission) 

Ya existe una columna para eso, pero algunos datos tienen la cantidad de marchas que tiene la transmisión (6at o AT8)

In [48]:
def remove_transmission(text):
    if not isinstance(text, str):
        return text
    # Remove patterns like "6at", "at4", "8mt", etc.
    text = re.sub(r'\b\d*\s*(at|mt|dct|cvt)\d*\b', '', text, flags=re.IGNORECASE)
    # Remove standalone transmission types and abbreviations
    text = re.sub(r'\b(at|mt|dct|cvt|automática|manual)\b', '', text, flags=re.IGNORECASE)
    return re.sub(' +', ' ', text).strip()

dataset['Versión'] = dataset['Versión'].apply(remove_transmission)

### Borrar los caballos de fuerza (cv o hp)

In [49]:
def remove_horsepower(text):
    if not isinstance(text, str):
        return text
    return re.sub(r'\b\d+\s*(cv|hp)\b', '', text, flags=re.IGNORECASE).replace('  ', ' ').strip()

# Apply to your dataset
dataset['Versión'] = dataset['Versión'].apply(remove_horsepower)

### Borrar la tracción

4x2 o 4x4

In [50]:
def remove_traction(text):
    if not isinstance(text, str):
        return text
    # Remove '4x4', '4x2', 'awd', '4wd', '2wd', 'fwd' (case-insensitive, x can be X)
    text = re.sub(r'\b4[xX][24]\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bawd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b4wd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b2wd\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bfwd\b', '', text, flags=re.IGNORECASE)
    return re.sub(' +', ' ', text).strip()

dataset['Versión'] = dataset['Versión'].apply(remove_traction)

# Resultado final

## Análisis de versiones

In [51]:
marca_ejemplo = 'Ford'
modelo_ejemplo = 'Ecosport'

cols_to_keep = ['idx','Marca', 'Modelo', 'Versión', 'Motor', 'cv', 'Tracción', 'Turbo']

new_filtered = dataset[(dataset['Marca'] == marca_ejemplo) & (dataset['Modelo'] == modelo_ejemplo)]
old_filtered = raw_dataset[(raw_dataset['Marca'] == marca_ejemplo) & (raw_dataset['Modelo'] == modelo_ejemplo)]

new_version_counts = (
    new_filtered.groupby(['Marca', 'Modelo', 'Versión'])
    .size()
    .reset_index(name='Cantidad')
)
new_version_counts.to_csv(f"data/examples/{marca_ejemplo}_{modelo_ejemplo}_new_versions.csv", index=False)

old_version_counts = (
    old_filtered.groupby(['Marca', 'Modelo', 'Versión'])
    .size()
    .reset_index(name='Cantidad')
)
old_version_counts.to_csv(f"data/examples/{marca_ejemplo}_{modelo_ejemplo}_old_versions.csv", index=False)

print(f"Versiones nuevas para {marca_ejemplo} {modelo_ejemplo}:\n")
print(f"Cantidad de versiones únicas en raw: {len(old_version_counts)}")
print(f"Cantidad de versiones únicas despues de filtrar: {len(new_version_counts)}")



Versiones nuevas para Ford Ecosport:

Cantidad de versiones únicas en raw: 110
Cantidad de versiones únicas despues de filtrar: 65


# Guardar en CSV

In [52]:

dataset.to_csv("data/cleaned.csv", index=False)